# Oracles and Smart Contracts — The Notary Learns to Fill In a Form

We have been building one public proof-of-stake chain. Notebook 2 gave it validators and a stake-weighted proposer. Notebook 5 gave it a waiting room: each node has its own mempool, gossip is imperfect, and a proposer can only include what it has heard. The chain is still a **notary**. It stamps what it is handed. It does not wander outside for the price of ETH.

A **smart contract** is the form the notary agrees to fill in: if a transaction names this address and this method, run these rules. Anyone can gossip a call. A PoS proposer includes it if they have heard it.

**Question:** if a contract can only see its own storage and the arguments you pass it, how does a lending protocol know what collateral is worth?

**Scope:** a deterministic teaching model, not a recipe for shipping or attacking a real protocol. No bytecode, no gas schedule, no lawyers. This is the public PoS chain from notebooks 2 and 5, now running code.


## Recap

Public proof-of-stake from notebook 2, plus the waiting room from notebook 5.

| Notebook | What it established |
| --- | --- |
| 2 | Validators, a stake-weighted proposer, slashing as hostage. |
| 5 | Each node has its own mempool. `broadcast`, then `include`. The chain is a notary, not a psychic. |

We **import** those classes. We do not reinvent `Block`, `Blockchain`, `Validator`, `Transaction`, or `Network`. We give them something to run.

The new classes below — `SmartContract`, `AMMPool`, `LendingProtocol`, `MedianOracle` — are defined inline so the lesson stays in one place. Notebook 7 imports the same copy from [`blockchain_lib/contracts.py`](../blockchain_lib/contracts.py). Same handoff as mempool after notebook 5.


## 1. What a smart contract actually is

The chain does not understand "loan" or "swap". It understands: this address, this method, these arguments. On Ethereum the rules would be bytecode. Here they are Python, because we are trying to remember an idea, not compile Solidity in a cafe.

`SmartContract` is deliberately thin: an `address`, and `call(method, **kwargs)` which runs a public method by name. Private names (`_...`) are not part of the surface. Two contracts will inherit it — a swap pool and a lending protocol — because those are the forms we will keep filling in all the way into notebook 7.

> Pause and predict: if the notary fills in the form correctly, does that mean the form was a good idea?


In [ ]:
import random
import statistics
from dataclasses import dataclass

from blockchain_lib.mempool import Network, Transaction
from blockchain_lib.pos import Block, Blockchain, Validator


class SmartContract:
    """Code that lives at an address and runs when something calls it.

    The chain does not understand loans or swaps. It understands: this
    address, this method, these arguments. The methods below are the rules.
    """

    def __init__(self, address: str) -> None:
        if not address:
            raise ValueError("Contract address must be non-empty.")
        self.address = address

    def call(self, method: str, **kwargs):
        """Invoke a public method by name."""
        if method.startswith("_") or method == "call":
            raise AttributeError(
                f"{self.address} has no public method {method!r}."
            )
        func = getattr(self, method, None)
        if not callable(func):
            raise AttributeError(  # noqa: TRY004
                f"{self.address} has no public method {method!r}."
            )
        return func(**kwargs)


def submit_call(network, chain, origin, tx_id, contract, method, **kwargs):
    """Gossip a contract call, include it, then run it.

    Inclusion is the notary stamp. ``call`` is the form being filled in.
    A proposer who never heard the gossip cannot stamp it — notebook 5
    still applies.
    """
    arg_preview = ", ".join(f"{k}={v!r}" for k, v in kwargs.items())
    description = f"{contract.address}.{method}({arg_preview})"
    network.broadcast(Transaction(tx_id, description), origin=origin)
    tx, block = network.include(origin, tx_id, chain)
    result = contract.call(method, **kwargs)
    return result, tx, block


## 2. The swap contract: a puddle with a price

A constant-product AMM keeps `x * y = k`. Here `x` is ETH reserve and `y` is USD reserve, so the displayed spot price is `USD reserve / ETH reserve`. That number is not a journalist. It is the current ratio of two piles of tokens. We omit fees, slippage limits, and external arbitrage so the splash is visible.

This is a **smart contract**: it has an address, and a swap is a method call.

We start with 50 ETH and $100,000 ($2,000/ETH) and sell 1 ETH and 40 ETH into **separate fresh pools**. A pebble and a boulder, same puddle, different splash.

> Pause and predict: which sale moves the displayed price more, and does `x * y` remain essentially unchanged?


In [6]:
class AMMPool(SmartContract):
    """A constant-product (x*y=k) two-asset market maker."""

    def __init__(
        self, address: str, eth_reserve: float, usd_reserve: float
    ) -> None:
        super().__init__(address)
        if eth_reserve <= 0 or usd_reserve <= 0:
            raise ValueError("AMM reserves must be positive.")
        self.eth_reserve = eth_reserve
        self.usd_reserve = usd_reserve

    @property
    def spot_price(self) -> float:
        """Current displayed price: USD reserve per unit of ETH reserve."""
        return self.usd_reserve / self.eth_reserve

    @property
    def constant_product(self) -> float:
        """The invariant ``x * y`` a swap should preserve (no fees here)."""
        return self.eth_reserve * self.usd_reserve

    def swap_eth_for_usd(self, eth_in: float) -> float:
        """Sell ETH into the pool, moving both reserves and the spot price."""
        if eth_in <= 0:
            raise ValueError("Swap input must be positive.")
        k = self.constant_product
        new_eth_reserve = self.eth_reserve + eth_in
        new_usd_reserve = k / new_eth_reserve
        usd_out = self.usd_reserve - new_usd_reserve
        self.eth_reserve, self.usd_reserve = new_eth_reserve, new_usd_reserve
        return usd_out

    def swap_usd_for_eth(self, usd_in: float) -> float:
        """Sell USD into the pool, moving both reserves and the spot price."""
        if usd_in <= 0:
            raise ValueError("Swap input must be positive.")
        k = self.constant_product
        new_usd_reserve = self.usd_reserve + usd_in
        new_eth_reserve = k / new_usd_reserve
        eth_out = self.eth_reserve - new_eth_reserve
        self.usd_reserve, self.eth_reserve = new_usd_reserve, new_eth_reserve
        return eth_out


In [7]:
initial_pool = AMMPool("amm-initial", 50.0, 100_000.0)
small_trade_pool = AMMPool("amm-pebble", 50.0, 100_000.0)
large_trade_pool = AMMPool("amm-boulder", 50.0, 100_000.0)

small_usd_out = small_trade_pool.call("swap_eth_for_usd", eth_in=1.0)
large_usd_out = large_trade_pool.call("swap_eth_for_usd", eth_in=40.0)
small_impact = (small_trade_pool.spot_price / initial_pool.spot_price - 1) * 100
large_impact = (large_trade_pool.spot_price / initial_pool.spot_price - 1) * 100
constant_product_preserved = abs(
    large_trade_pool.constant_product - initial_pool.constant_product
) < 1e-6

print(
    f"Initial pool: {initial_pool.eth_reserve:.2f} ETH and "
    f"${initial_pool.usd_reserve:,.2f}; spot price ${initial_pool.spot_price:,.2f}/ETH"
)
print(
    f"Small 1 ETH sale: receives ${small_usd_out:,.2f}; "
    f"price ${small_trade_pool.spot_price:,.2f}/ETH ({small_impact:.2f}%)"
)
print(
    f"Large 40 ETH sale: receives ${large_usd_out:,.2f}; "
    f"price ${large_trade_pool.spot_price:,.2f}/ETH ({large_impact:.2f}%)"
)
print(f"x * y preserved within floating-point tolerance: {constant_product_preserved}")


Initial pool: 50.00 ETH and $100,000.00; spot price $2,000.00/ETH
Small 1 ETH sale: receives $1,960.78; price $1,922.34/ETH (-3.88%)
Large 40 ETH sale: receives $44,444.44; price $617.28/ETH (-69.14%)
x * y preserved within floating-point tolerance: True


**Read the result:** the 40 ETH sale moves the displayed price by about 69%, far more than the 1 ETH sale. Nothing says ETH became 69% cheaper everywhere; this thin pool observed its own reserves. The AMM is an honest reporter of a local puddle. The constant product stays stable apart from ordinary floating-point rounding.

Remember **$617/ETH** and "a boulder in a puddle". Notebook 7 will rent the boulder.


## 3. The lending contract: a loan that asks the puddle

The protocol liquidates whenever collateral value / debt falls below 1.5. Its mistake is intentionally narrow: it reads `pool.spot_price` directly. That is an **oracle** choice, even though nobody named a courier. A contract cannot squint at an exchange. Whatever it treats as the price *is* its oracle.

Meet the loan we will keep bullying: **10 ETH collateral, $12,000 debt**. At a sensible ~$2,000/ETH the collateral ratio is 1.67.

Real protocols also wrestle with bonuses, partial liquidations, fees, and token transfers. We omit those so the bad price source is the only moving part.

> Pause and predict: at $2,000/ETH, is this position above or below a 1.5 threshold?


In [10]:
@dataclass(frozen=True)
class Loan:
    """A borrower's position: collateral posted against debt owed."""

    collateral_eth: float
    debt_usd: float

    def __post_init__(self) -> None:
        if self.collateral_eth <= 0 or self.debt_usd <= 0:
            raise ValueError("Loan collateral and debt must be positive.")


class LoanNotFoundError(Exception):
    """Raised when a borrower has no open loan."""


class PositionNotLiquidatableError(Exception):
    """Raised when liquidation is attempted on a still-healthy position."""


class LendingProtocol(SmartContract):
    """A toy lending protocol that liquidates undercollateralised loans.

    Its one deliberate flaw: by default it prices collateral from
    ``pool.spot_price`` unless a ``price_source`` is supplied instead.
    """

    def __init__(
        self,
        address: str,
        pool: AMMPool,
        liquidation_ratio: float = 1.5,
        price_source=None,
    ) -> None:
        super().__init__(address)
        if liquidation_ratio <= 0:
            raise ValueError("Liquidation ratio must be positive.")
        self.pool = pool
        self.liquidation_ratio = liquidation_ratio
        self.price_source = price_source
        self.loans: dict[str, Loan] = {}

    def open_loan(
        self, borrower: str, collateral_eth: float, debt_usd: float
    ) -> Loan:
        """Open a loan for ``borrower`` and return it."""
        return self.add_loan(borrower, Loan(collateral_eth, debt_usd))

    def add_loan(self, borrower: str, loan: Loan) -> Loan:
        """Register an open loan for a borrower."""
        if not borrower:
            raise ValueError("Borrower name must be non-empty.")
        self.loans[borrower] = loan
        return loan

    def collateral_ratio(self, loan: Loan) -> float:
        """Return collateral value divided by debt, using the configured price source."""
        price = (
            self.price_source.price
            if self.price_source is not None
            else self.pool.spot_price
        )
        return loan.collateral_eth * price / loan.debt_usd

    def liquidate(self, borrower: str) -> Loan:
        """Seize and remove a borrower's loan if it is unhealthy."""
        if borrower not in self.loans:
            raise LoanNotFoundError(f"No loan for {borrower}.")
        loan = self.loans[borrower]
        if self.collateral_ratio(loan) >= self.liquidation_ratio:
            raise PositionNotLiquidatableError("Position is still healthy.")
        return self.loans.pop(borrower)


In [11]:
baseline_pool = AMMPool("amm-baseline", 50.0, 100_000.0)
baseline_protocol = LendingProtocol("lending-baseline", baseline_pool)
victim_loan = baseline_protocol.call(
    "open_loan", borrower="victim", collateral_eth=10.0, debt_usd=12_000.0
)
baseline_ratio = baseline_protocol.collateral_ratio(victim_loan)

print(f"At ${baseline_pool.spot_price:,.2f}/ETH, victim collateral ratio: {baseline_ratio:.2f}")
print("Liquidation threshold: 1.50; verdict: HEALTHY")


At $2,000.00/ETH, victim collateral ratio: 1.67
Liquidation threshold: 1.50; verdict: HEALTHY


**Read the result:** $20,000 of collateral / $12,000 debt is 1.67, so the position is healthy before anyone touches the pool. A later liquidation would not be the victim changing their loan. It would be the protocol trusting a temporary photograph of the puddle.


## 4. Put it on the chain we already built

Same `Validator`, `Blockchain`, `Network`, and `Transaction` as notebooks 2 and 5. We instantiate an AMM at `amm.eth` and a lending protocol at `lending.eth`, then gossip two calls: Bob opens the 10 ETH / $12,000 loan, Alice swaps 1 ETH.

`submit_call` stamps the payload, then runs the method. A node that missed the gossip still cannot include it.

> Pause and predict: after Alice's 1 ETH swap, is Bob still above 1.5?


In [14]:
nodes = ["Node A", "Alice-Node", "Bob-Node", "Farid-Node"]
validators = [
    Validator("Node A", 100),
    Validator("Alice-Node", 80),
    Validator("Bob-Node", 70),
    Validator("Farid-Node", 50),
]
network = Network(nodes, random.Random(7))
chain = Blockchain(validators)

amm = AMMPool("amm.eth", 50.0, 100_000.0)
lending = LendingProtocol("lending.eth", amm)

network.broadcast(Transaction("tx-deploy-amm", "Deploy AMMPool at amm.eth"), origin="Node A")
network.broadcast(Transaction("tx-deploy-lending", "Deploy LendingProtocol at lending.eth"), origin="Node A")
_, deploy_amm_block = network.include("Node A", "tx-deploy-amm", chain)
_, deploy_lending_block = network.include("Node A", "tx-deploy-lending", chain)

bob_loan, bob_tx, bob_block = submit_call(
    network,
    chain,
    "Bob-Node",
    "tx-open-loan",
    lending,
    "open_loan",
    borrower="Bob",
    collateral_eth=10.0,
    debt_usd=12_000.0,
)
alice_usd, alice_tx, alice_block = submit_call(
    network,
    chain,
    "Alice-Node",
    "tx-swap-1eth",
    amm,
    "swap_eth_for_usd",
    eth_in=1.0,
)
bob_ratio_after_pebble = lending.collateral_ratio(lending.loans["Bob"])
valid, message = chain.is_valid()

print(f"Deployed {amm.address} in Block #{deploy_amm_block.index}")
print(f"Deployed {lending.address} in Block #{deploy_lending_block.index}")
print(f"{bob_block.proposer} included {bob_tx.tx_id} in Block #{bob_block.index}")
print(f"{alice_block.proposer} included {alice_tx.tx_id} in Block #{alice_block.index}")
print(
    f"After Alice's 1 ETH swap: pool spot ${amm.spot_price:,.2f}/ETH; "
    f"Bob's ratio {bob_ratio_after_pebble:.2f}"
)
print(f"Chain valid? {valid} -- {message}")
print(f"Canonical length: {len(chain.chain)} (genesis + {len(chain.chain) - 1} inclusions)")


Deployed amm.eth in Block #1
Deployed lending.eth in Block #2
Bob-Node included tx-open-loan in Block #3
Alice-Node included tx-swap-1eth in Block #4
After Alice's 1 ETH swap: pool spot $1,922.34/ETH; Bob's ratio 1.60
Chain valid? True -- Chain is valid.
Canonical length: 5 (genesis + 4 inclusions)


**Read the result:** Bob's loan and Alice's swap are ordinary notebook-5 transactions. Alice's pebble moves the puddle a little (~$1,922/ETH) and Bob stays healthy at about 1.60. The notary did its job. The form it filled in still asks the puddle what ETH is worth.


## 5. Oracles, quickly

A smart contract can inspect its own state and the inputs you hand it. It cannot check an exchange screen. Someone has to *tell* it. Consensus notarises what it was given. It does not fact-check the universe.

An **oracle** is not a third contract in this model. It is the input the lending contract treats as fact. That input might be a named feed, a median of feeds, or `amm.spot_price`. The last one already lives on-chain, which makes it tempting, and that is the trap.

Three things an oracle is *not*, even when the number looks official:

- **Authenticated.** A `source` string is a label, not a signature. Nothing here proves Exchange A sent the report.
- **Fresh.** A correct price from last Tuesday is still a wrong input today. This toy has no heartbeat.
- **Trustworthy just because it is on-chain.** `amm.spot_price` is readable by the contract. Readable is not the same as "the market."

Median aggregation raises the cost of one liar. It does not timestamp the reports, and it does not help if the protocol later ignores the median and reads a thin pool instead.

Same loan: 10 ETH / $12,000. Feed it a liar, then a median.

> Pause and predict: if the contract trusts one reported price of $1,200, what happens to this loan? What if that lie sits next to two reports near $2,000?


In [17]:
@dataclass(frozen=True)
class PriceReport:
    """One reported ETH/USD price from a named source.

    Attributes:
        source: Who reported this price. Nothing here authenticates them.
        eth_price_usd: The reported price.
    """

    source: str
    eth_price_usd: float


def collateral_ratio(
    collateral_eth: float, debt_usd: float, eth_price_usd: float
) -> float:
    """Return collateral value divided by debt, at a given ETH price."""
    if collateral_eth <= 0 or debt_usd <= 0 or eth_price_usd <= 0:
        raise ValueError("Collateral, debt, and price must be positive.")
    return collateral_eth * eth_price_usd / debt_usd


def should_liquidate(ratio: float, threshold: float = 1.5) -> bool:
    """Return whether a collateral ratio is below the liquidation threshold."""
    return ratio < threshold


class MedianOracle:
    """A price source that reports the median of several independent reports."""

    def __init__(self, reports: list[PriceReport]) -> None:
        if not reports:
            raise ValueError("At least one price report is required.")
        self.reports = reports

    @property
    def price(self) -> float:
        """Median reported price — resists a single outlier report."""
        return statistics.median(report.eth_price_usd for report in self.reports)


In [18]:
collateral_eth = 10
debt_usd = 12_000
bad_report = PriceReport("malicious-feed", 1_200)
bad_report_ratio = collateral_ratio(
    collateral_eth, debt_usd, bad_report.eth_price_usd
)

print(f"Position: {collateral_eth} ETH collateral / ${debt_usd:,} debt")
print(f"Single report: ${bad_report.eth_price_usd:,.0f}/ETH")
print(f"Single-source collateral ratio: {bad_report_ratio:.2f}")
if should_liquidate(bad_report_ratio):
    print("Single-source verdict: LIQUIDATE (wrong)")

reports = [
    PriceReport("independent-feed-a", 2_005),
    PriceReport("malicious-feed", 1_200),
    PriceReport("independent-feed-b", 1_995),
]
median_oracle = MedianOracle(reports)
median_price = median_oracle.price
median_report_ratio = collateral_ratio(collateral_eth, debt_usd, median_price)

print()
print("Reports: $2,005, $1,200, $1,995 per ETH")
print(f"Median report: ${median_price:,.0f}/ETH")
print(f"Median collateral ratio: {median_report_ratio:.2f}")
if not should_liquidate(median_report_ratio):
    print("Median verdict: HEALTHY")


Position: 10 ETH collateral / $12,000 debt
Single report: $1,200/ETH
Single-source collateral ratio: 1.00
Single-source verdict: LIQUIDATE (wrong)

Reports: $2,005, $1,200, $1,995 per ETH
Median report: $1,995/ETH
Median collateral ratio: 1.66
Median verdict: HEALTHY


**Read the result:** the false $1,200 price values 10 ETH at $12,000, so the ratio is 1.00 and the rule liquidates a healthy position. Correct arithmetic, wrong universe. The median ignores the outlier, keeps $1,995, and the ratio 1.66 stays healthy.

Aggregation raises the cost of lying. It does not prove the feeds were signed, or fresh, or that the protocol will actually read them. On-chain readable is not the same as trustworthy. A puddle is not the ocean.


## 6. You can shove the puddle — if you brought your own ETH

No lying courier required. The pool will tell the truth about *itself*, and that truth will be a terrible proxy for "the market."

Alice has **40 ETH of her own**. Same boulder you already watched. We use a fresh 50 ETH puddle so the arithmetic matches section 2, then dump and liquidate as **two transactions**. Between them the cheap price sits on-chain, in public, the way notebook 5 said a waiting room works.

This is not a flash loan. Alice had to be rich *before* lunch.

> Pause and predict: after paying the $12,000 debt and buying ETH back, how much of Alice's original 40 ETH remains — and did she need to own those 40 ETH the whole time?


In [21]:
# Fresh puddle so the 40 ETH splash matches section 2.
attack_pool = AMMPool("amm-thin.eth", 50.0, 100_000.0)
attack_lending = LendingProtocol("lending-thin.eth", attack_pool)
attack_lending.call(
    "open_loan", borrower="victim", collateral_eth=10.0, debt_usd=12_000.0
)

alice_eth = 40.0
usd_from_dump, dump_tx, dump_block = submit_call(
    network,
    chain,
    "Alice-Node",
    "tx-dump-40eth",
    attack_pool,
    "swap_eth_for_usd",
    eth_in=alice_eth,
)
manipulated_price = attack_pool.spot_price
victim_ratio = attack_lending.collateral_ratio(attack_lending.loans["victim"])
seized, liq_tx, liq_block = submit_call(
    network,
    chain,
    "Alice-Node",
    "tx-liquidate-victim",
    attack_lending,
    "liquidate",
    borrower="victim",
)
usd_for_buyback = usd_from_dump - seized.debt_usd
eth_bought_back = attack_pool.call("swap_usd_for_eth", usd_in=usd_for_buyback)
alice_final_eth = eth_bought_back + seized.collateral_eth
profit_eth = alice_final_eth - alice_eth

print(f"{dump_block.proposer} included {dump_tx.tx_id} in Block #{dump_block.index}")
print(
    f"DUMP: Alice sells {alice_eth:.2f} ETH of her own for ${usd_from_dump:,.2f}; "
    f"AMM price becomes ${manipulated_price:,.2f}/ETH."
)
print(
    f"The cheap puddle is now public. Victim ratio at this price: {victim_ratio:.2f}."
)
print(f"{liq_block.proposer} included {liq_tx.tx_id} in Block #{liq_block.index}")
print(
    f"LIQUIDATE: pay ${seized.debt_usd:,.2f} debt and seize "
    f"{seized.collateral_eth:.2f} ETH."
)
print(
    f"BUY BACK: remaining ${usd_for_buyback:,.2f} buys {eth_bought_back:.2f} ETH."
)
print(
    f"Alice started with {alice_eth:.2f} ETH, ends with {alice_final_eth:.2f} ETH, "
    f"profit {profit_eth:.2f} ETH."
)
print("She had to own the 40 ETH already. The bug is the price source, not the financing.")


Alice-Node included tx-dump-40eth in Block #5
DUMP: Alice sells 40.00 ETH of her own for $44,444.44; AMM price becomes $617.28/ETH.
The cheap puddle is now public. Victim ratio at this price: 0.51.
Alice-Node included tx-liquidate-victim in Block #6
LIQUIDATE: pay $12,000.00 debt and seize 10.00 ETH.
BUY BACK: remaining $32,444.44 buys 33.18 ETH.
Alice started with 40.00 ETH, ends with 43.18 ETH, profit 3.18 ETH.
She had to own the 40 ETH already. The bug is the price source, not the financing.


**Read the result:** the dump produces $44,444.44 and makes the victim look unsafe at a 0.51 ratio. Paying the $12,000 debt leaves $32,444.44 for the reverse swap, which buys back about 33.18 ETH. Add the seized 10 ETH collateral, subtract the 40 Alice started with, and about **3.18 ETH** remains as profit.

Every step followed the rules. The rules asked the puddle for the price of the ocean. The dump and the liquidation were *two* transactions, so the cheap price sat in the public waiting room between them. In this toy nobody snipes it. On a real network, somebody might.

Two things to carry into notebook 7:

1. **The bug is the oracle choice.** Alice did not break arithmetic. She fed the lending contract a local ratio and it believed her.
2. **The financing is still a suitcase of cash.** Alice needed 40 ETH sitting around. A flash loan is what happens when you rent the suitcase for one elevator ride, and dump + liquidate + repay become *one* atomic transaction.

Continue with [7. flash_loans.ipynb](7.%20flash_loans.ipynb).


## Takeaways

- **Mechanism:** a smart contract is code the notary will run when a transaction names its address. **Not a guarantee:** the rules were a good idea.
- **Mechanism:** `broadcast`, then `include`, then `call` — same waiting room as notebook 5. **Not a guarantee:** a missed gossip still lets you include it.
- **Mechanism:** an AMM spot price honestly reflects its own reserves. **Not a guarantee:** that local ratio is "the market."
- **Mechanism:** an oracle is the input a contract treats as a fact about the world. **Not a guarantee:** that fact is true.
- **Mechanism:** a median of independent reports raises the cost of one liar. **Not a guarantee:** aggregation proves reality, or saves you if you still read a thin AMM.
- **Mechanism:** you can shove a thin pool with capital you already own. **Not a guarantee:** you needed to be rich first, or that two separate transactions will stay unopposed. Notebook 7 removes both of those comforts.
